In [ ]:
import jax
import numpy as np
import flax.linen as nn

import matplotlib.pyplot as plt
import h5py as hf
from dataclasses import dataclass
import seaborn as sns
from mpl_toolkits.mplot3d import axes3d
from matplotlib.lines import Line2D
import pint
from matplotlib.colors import LogNorm
import matplotlib.tri as tri

In [ ]:
@dataclass
class Measurement:
    data: np.ndarray
    uncertainty: np.ndarray

In [ ]:
def load_data(
    temperature: int, speed: int, radius: float, ensembles: int = 10
):
    """
    Load data from a specific experiment and average over ensembles.
    """
    root_path = f"{temperature}/{speed}/{radius}mum"
    distances = []
    rewards = []
    
    for i in range(1, 21):
        try:
            with hf.File(f"{root_path}/{i}/deployment/trajectory.hdf5", "r") as db:
                data = db["colloids"]["Unwrapped_Positions"][:]
                distance_to_source = np.linalg.norm(data - np.array([500., 500., 0.]), axis=-1)
                distances.append(
                    Measurement(
                        data=np.mean(distance_to_source, axis=1),
                        uncertainty=np.std(distance_to_source, axis=1)
                    )
                    
                )
                reward = (
                    np.sum(np.load(f"{root_path}/{i}/rewards_exploration.npy", allow_pickle=True)) + 
                    np.sum(np.load(f"{root_path}/{i}/rewards_no_exploration.npy", allow_pickle=True))
                )
                rewards.append(reward)
        except:
            pass
    
    
    return distances, rewards

In [ ]:
sizes = np.arange(0.025, 2.51, 0.075)
temperature=300
speeds = np.arange(1, 5.5, 0.5)

## Training Ability

In [ ]:
results = {}

for speed in speeds:
    results[speed] = {}
    for size in sizes:
        data, rewards = load_data(temperature=temperature, speed=speed, radius=f"{size:0.3f}")
        results[speed][size] = []

        for i, distances in enumerate(data):
            if np.mean(distances.data[500:]) < 15:
                if rewards[i] == 0.0:
                    new = 0.0
                else:
                    distance = 1 / rewards[i]
                    distance /= (size / 1000)
                    new = 1 / distance
                results[speed][size].append(new)

        results[speed][size]

In [ ]:
data_points = np.zeros((len(speeds), len(sizes)))
his_points = []

for i, item in enumerate(speeds[::-1]):
    
    x = list(results[item].keys())
    y = list(results[item].values())
    
    for j, size in enumerate(x):
        data_points[i, j] = np.mean(y[j])
        his_points.append([size, item, np.mean(y[j])])
    
his_points = np.array(his_points)

In [ ]:
Z = np.zeros_like(X)

for t, i in enumerate(Y):
    for j in X:
        for p, k in enumerate(j):
            Z[t, p] = np.mean(results[i[0]][k])

In [ ]:
# quadric, gaussian, bessel, mitchell, lanczos
# fig, ax = plt.subplots(1, 2, figsize=(10, 5))

# Scatter plot
plt.scatter(
    his_points[:, 0], 
    his_points[:, 1], 
    c=np.nan_to_num(his_points[:, 2]),
    cmap="Spectral"
)

# X, Y = np.meshgrid(his_points[:, 0], his_points[:, 1])
# # _, Z = np.meshgrid(his_points[:, 0], his_points[:, 2])


# im = ax[1].pcolormesh(X, Y, np.nan_to_num(Z), cmap="Spectral", shading="gouraud")
# cbar = fig.colorbar(im)
plt.xlabel(r"Colloid Radius / $\mu m$")
# ax[1].set_xlabel(r"Colloid Radius / $\mu m$")
plt.ylabel(r"Swim Speed / body lengths $s^{-1}$")
# cbar.set_label("Total reward from training")
cbar = plt.colorbar()
cbar.set_label("Total reward from training")

plt.savefig("reward-phase-diagram.pdf")
plt.show()

# Rate of Success

We check if the last 500 points are les than 30 micro-meters from the source and if so, consider the model as a success.

In [ ]:
results = {}

for speed in speeds:
    results[speed] = {}
    for size in sizes:
        data, _ = load_data(temperature=temperature, speed=speed, radius=f"{size:0.3f}")

        results[speed][size] = 0

        for distances in data:
            if np.mean(distances.data[500:]) < 15:
                results[speed][size] += 1

        results[speed][size] /= 20
        

In [ ]:

data_points = np.zeros((len(speeds), len(sizes)))
his_points = []

for i, item in enumerate(speeds[::-1]):
    
    x = list(results[item].keys())
    y = list(results[item].values())
    
    for j, size in enumerate(x):
        data_points[i, j] = y[j]
        his_points.append([size, item, y[j]])
    
his_points = np.array(his_points)

In [ ]:
# quadric, gaussian, bessel, mitchell, lanczos
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

# Scatter plot
ax[0].scatter(his_points[:, 0], his_points[:, 1], c=his_points[:, 2], cmap="Blues")

# Image plot
img = ax[1].imshow(
    data_points, 
    extent=[0.025, 2.5, 1.0, 5.0], 
    aspect="auto", 
    vmin=0.,
    vmax=1.,
    interpolation="gaussian",
    cmap="PuBu"
)

ax[0].set_xlabel(r"Colloid Radius / $\mu m$")
ax[1].set_xlabel(r"Colloid Radius / $\mu m$")

ax[0].set_ylabel(r"Swim Speed / body lengths $s^{-1}$")
cbar = fig.colorbar(img)
cbar.set_label("Probability of emergent chemotaxis")
plt.savefig("probability-phase-diagram.pdf")
plt.show()

# Policy Efficiency

Here we plot the average distance for the last 500 steps as well as the rate at which the colloids reach the center for each size.

## Final distance

In [ ]:
distance_results = {}

for speed in speeds:
    distance_results[speed] = {}
    for size in sizes:
        data, _ = load_data(temperature=temperature, speed=speed, radius=f"{size:0.3f}")

        distance_array = []

        for distances in data:
            if np.mean(distances.data[500:]) < 15:
                distance_array.append(np.mean(distances.data[500:]))

        distance_results[speed][size] = Measurement(
            data=np.mean(distance_array),
            uncertainty=np.std(distance_array)
        )

convergence_results = {}

for speed in speeds:
    convergence_results[speed] = {}
    for size in sizes:
        data, _ = load_data(temperature=temperature, speed=speed, radius=f"{size:0.3f}")

        convergence_time = []

        for distances in data:
            if np.mean(distances.data[500:]) < 15 and max(distances.data) < 100:
                convergence_time.append(np.where(distances.data <=10.0)[0][0])

        convergence_results[speed][size] = Measurement(
            data=np.mean(convergence_time, axis=0),
            uncertainty=np.std(convergence_time, axis=0)
        )

In [ ]:
distance_results.keys()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

linestyles = [
    '-', '--', '-.', ':', (5, (10, 3))
]
markers = [
    ".", "o", "v", "1", "s"
]
colours = [
    "#433A3F",
    "#3D5A6C",
    "#72A98F", 
    "#8DE969", 
    "#FF4A1C"
]

j = 0
for i, item in enumerate(distance_results):
    if i % 2 == 0:
        x_values = distance_results[item].keys()
        y_values = distance_results[item].values()

        ax[0].errorbar(
            x=list(x_values), 
            y=[item.data for item in y_values], 
            yerr=[item.uncertainty for item in y_values],
            marker=markers[j],
            capsize=8,
            ls=linestyles[j],
            label=item,
            c=colours[j]
        )
        j += 1

        
j = 0
for i, item in enumerate(convergence_results):
    if i % 2 == 0:
        convergence_rate = []
        sizes = []
        errors = []
        for size in convergence_results[item]:
            try:
#                 index = np.where(convergence_results[item][size].data <= 10.0)[0][0]
                convergence_rate.append(convergence_results[item][size])
                sizes.append(size)
            except:
                continue

        ax[1].errorbar(
            x=sizes, 
            y=[item.data * 0.1 for item in convergence_rate],
            yerr=[item.uncertainty * 0.1 for item in convergence_rate],
            marker=markers[j],
            ls=linestyles[j],
            capsize=8,
            c=colours[j],
#             label=item
        )
        j += 1
    else:
        pass

ax[0].set_xlabel(r"Colloid Radius / $\mu m$")
ax[0].set_ylabel(r"Equilibrium Distance from Source / $\mu m$")

ax[1].set_xlabel(r"Colloid Radius / $\mu m$")
ax[1].set_ylabel(r"Time to minimum / s")

fig.legend(title="Swim Speed", loc=[0.8, 0.5])

plt.savefig("policy-efficacy.pdf")
plt.show()


## Convergence Rate 

In [ ]:
convergence_results = {}

for speed in speeds:
    convergence_results[speed] = {}
    for size in sizes:
        data, _ = load_data(temperature=temperature, speed=speed, radius=f"{size:0.3f}")

        distance_array = []

        for distances in data:
            if np.mean(distances.data[500:]) < 15 and max(distances.data) < 100:
                distance_array.append(distances.data)

        convergence_results[speed][size] = Measurement(
            data=np.mean(distance_array, axis=0),
            uncertainty=np.std(distance_array, axis=0)
        )

In [ ]:
linestyles = [
    '-', '--', '-.', ':', (5, (10, 3))
]
markers = [
    ".", "o", "v", "1", "s"
]
colours = [
    "#433A3F",
    "#3D5A6C",
    "#72A98F", 
    "#8DE969", 
    "#FF4A1C"
]

j = 0
for i, item in enumerate(convergence_results):
    if i % 2 == 0:
        convergence_rate = []
        sizes = []
        errors = []
        for size in convergence_results[item]:
            try:
                index = np.where(convergence_results[item][size].data <= 10.0)[0][0]
                convergence_rate.append(index)
                sizes.append(size)
            except:
                continue

        plt.plot(
            sizes, 
            np.array(convergence_rate) * 0.1, 
            marker=markers[j],
            ls=linestyles[j],
            c=colours[j],
            label=item)
        j += 1
    else:
        pass

plt.xlabel(r"Colloid Radius / $\mu m$")
plt.ylabel(r"Time to minimum / s")
plt.legend(title="Swim Speed")
plt.savefig("convergence-speed.pdf")
plt.show()

# Policy Evaluation

Here we use the policy evaluation tool to compare learned approaches.

In [ ]:
results = {}

class ActorNet(nn.Module):
    """A simple dense model."""

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=4)(x)
        return x
    
model = ActorNet()
x = np.linspace(-1.7, 1.7, 100)

for speed in speeds:
    results[speed] = {}
    for size in sizes:
        data, _ = load_data(temperature=temperature, speed=speed, radius=f"{size:0.3f}")
        actions = []
        
        for i, item in enumerate(data):
            if np.mean(item.data[500:]) < 15 and max(item.data) < 100:
                parameters = np.load(
                    f"{temperature}/{speed}/{size:0.3f}mum/{i + 1}/Models/ActorModel_0.pkl",
                    allow_pickle=True
                )
                actions.append(
                        jax.nn.softmax(model.apply({"params": parameters[0]}, x.reshape(-1, 1))),
                )
            
        results[speed][f'{size:.3f}'] = Measurement(
            data=np.nanmean(actions, axis=0),
            uncertainty=np.nanstd(actions, axis=0)
        )

In [ ]:
colours = [
    "#177E89",
    "#084C61",
    "#DB3A34", 
    "#FFC857", 
    "#323031"
]

fig, ax = plt.subplots(5, 4, figsize=(15, 20), subplot_kw={"projection": '3d'})

k = 0
for j, speed in enumerate(speeds):
    if j % 2 == 0:
        X, Y = np.meshgrid(x, np.array(list(results[speed].keys()), dtype=float))
        Z_cw = np.zeros_like(Y)
        Z_tr = np.zeros_like(Y)
        Z_ccw = np.zeros_like(Y)
        Z_dn = np.zeros_like(Y)

        for i, item in enumerate(results[speed]):    
            try:
                Z_cw[i, :] = results[speed][item].data[:, 0]
            except:
                Z_cw[i, :] = np.zeros_like(Y[i])

            try:
                Z_tr[i, :] = results[speed][item].data[:, 1]
            except:
                Z_tr[i, :] = np.zeros_like(Y[i])

            try:
                Z_ccw[i, :] = results[speed][item].data[:, 2]
            except:
                Z_ccw[i, :] = np.zeros_like(Y[i])

            try:
                Z_dn[i, :] = results[speed][item].data[:, 3]
            except:
                Z_dn[i, :] = np.zeros_like(Y[i])


        ax[k, 0].plot_surface(
            X, Y, Z_cw, 
            edgecolor=colours[k], 
            lw=0.1,
            rstride=1, 
            cstride=1, 
            alpha=0.3,
            label="ahgh"
        )

        ax[k, 0].view_init(elev=20., azim=70.)

        ax[k, 1].plot_surface(
            X, Y, Z_tr, 
            edgecolor=colours[k], 
            lw=0.1,
            rstride=1, 
            cstride=1, 
            alpha=0.3,
        )

        ax[k, 1].view_init(elev=20., azim=70.)

        ax[k, 2].plot_surface(
            X, Y, Z_ccw, 
            edgecolor=colours[k], 
            lw=0.1,
            rstride=1, 
            cstride=1, 
            alpha=0.3,
        )

        ax[k, 2].view_init(elev=20., azim=70.)

        ax[k, 3].plot_surface(
            X, Y, Z_dn, 
            edgecolor=colours[k], 
            lw=0.1,
            rstride=1, 
            cstride=1, 
            alpha=0.3,
        )

        ax[k, 3].view_init(elev=20., azim=70.)

        if k == 0:
            ax[k, 0].set_title("CW")
            ax[k, 1].set_title("Translate")
            ax[k, 2].set_title("CCW")
            ax[k, 3].set_title("Do Nothing")


        ax[k, 0].set_ylabel(r"Colloid Size / $\mu m$")
        ax[k, 0].set_xlabel("Gradient Change")
        ax[k, 0].set_zlabel("Probability")

        ax[k, 1].set_ylabel(r"Colloid Size / $\mu m$")
        ax[k, 1].set_xlabel("Gradient Change")
        ax[k, 1].set_zlabel("Probability")

        ax[k, 2].set_ylabel(r"Colloid Size / $\mu m$")
        ax[k, 2].set_xlabel("Gradient Change")
        ax[k, 2].set_zlabel("Probability")

        ax[k, 3].set_ylabel(r"Colloid Size / $\mu m$")
        ax[k, 3].set_xlabel("Gradient Change")
        ax[k, 3].set_zlabel("Probability")
        
        k += 1

plt.tight_layout()


custom_lines = [
    Line2D([0], [0], color=colours[0], lw=4),
    Line2D([0], [0], color=colours[1], lw=4),
    Line2D([0], [0], color=colours[2], lw=4),
    Line2D([0], [0], color=colours[3], lw=4),
    Line2D([0], [0], color=colours[4], lw=4)
]

fig.legend(custom_lines, [1, 2, 3, 4, 5], title="Swim Speed", loc=(0.79, 0.88))


plt.savefig("policy.png", bbox_inches='tight', dpi=400)
plt.show()

## Biological Connection

In [ ]:
ureg = pint.UnitRegistry()

sizes = np.arange(0.025, 2.51, 0.075)
temperature=300
speeds = np.arange(1, 5.5, 0.5)

from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import MaxNLocator

In [ ]:
def compute_rotational_quantity(
    radius: ureg.Quantity, 
    rotational_velocity: float = ureg.Quantity(600.0, "degrees / second"),
    temperature: ureg.Quantity = ureg.Quantity(300.0, "kelvin")
):
    """
    """
    Dr = (ureg.boltzmann_constant * temperature) / (8 * np.pi * ureg.Quantity(8.9e-4, "pascal * second") * radius ** 3)
    
    tau_brownian =  1 / (2 * Dr)
    
    tau_active = np.pi / rotational_velocity
    
    return tau_brownian / tau_active
    

def compute_translation_quantity(
    radius: ureg.Quantity,
    velocity: ureg.Quantity,
    temperature: ureg.Quantity = ureg.Quantity(300.0, "kelvin")
):
    """
    """
    Dt = (ureg.boltzmann_constant * temperature) / (6 * np.pi * ureg.Quantity(8.9e-4, "pascal * second") * radius)
    
    tau_brownian = radius ** 2 / Dt
    
    tau_active =  2 * radius / velocity
    
    return (tau_brownian / tau_active).to("dimensionless").m

In [ ]:
ratios = []

for item in sizes:
    ratios.append(compute_rotational_quantity(radius=ureg.Quantity(item, "micrometers")).to("dimensionless"))


In [ ]:
plt.plot(sizes, ratios)
plt.yscale("log")

plt.xlabel("Radius")
# plt.ylabel("")

In [ ]:
data_points = np.zeros((len(speeds), len(sizes)))
his_points = []

for i, item in enumerate(speeds[::-1]):
    
    x = list(results[item].keys())
    y = list(results[item].values())
    
    for j, size in enumerate(x):
        data_points[i, j] = compute_translation_quantity(
            ureg.Quantity(float(size), "micrometers"),
            ureg.Quantity(item * 2 * float(size), "micrometers / second")
        )
        his_points.append([float(size), float(item), compute_translation_quantity(
            ureg.Quantity(float(size), "micrometers"),
            ureg.Quantity(item * 2 * float(size), "micrometers / second")
        )])
    
his_points = np.array(his_points)

In [ ]:
X, Y = np.meshgrid(np.linspace(0., 2.5, 50), np.linspace(0, 5.1, 50))

In [ ]:
Z = compute_translation_quantity(ureg.Quantity(X, "micrometers"), ureg.Quantity(2 * Y * X, "micrometers / second"))

In [ ]:
results = {}

for speed in speeds:
    results[speed] = {}
    for size in sizes:
        data, _ = load_data(temperature=temperature, speed=speed, radius=f"{size:0.3f}")

        results[speed][size] = 0

        for distances in data:
            if np.mean(distances.data[500:]) < 15:
                results[speed][size] += 1

        results[speed][size] /= 20
        

data_points = np.zeros((len(speeds), len(sizes)))
his_points = []

for i, item in enumerate(speeds[::-1]):
    
    x = list(results[item].keys())
    y = list(results[item].values())
    
    for j, size in enumerate(x):
        data_points[i, j] = y[j]
        his_points.append([size, item, y[j]])
    
his_points = np.array(his_points)

In [ ]:
plt.scatter(
    his_points[:, 0], 
    his_points[:, 1], 
    c=his_points[:, 2], 
    cmap="Blues"
)
plt.colorbar(label="Probability of Emergent Chemotaxis")

plt.contour(X, Y, Z, levels=list([1]), linewidths=3, colors="#8ACB88")

plt.vlines(0.48, 0.0, 5.1, lw=3, ls="--", color="#8ACB88")

plt.xlim(0, 2.6)
plt.ylim(0.9, 5.1)

plt.xlabel(r"Colloid Radius / $\mu m$")
plt.ylabel(r"Swim Speed / body length $s^{-1}$")

plt.savefig("probability-of-chemotaxis.pdf")
plt.show()

In [ ]:
n = 2 * ureg.boltzmann_constant * ureg.Quantity(300.0, "kelvin")
d = 8 * ureg.Quantity(8.9e-4, "pascal * second") * ureg.Quantity(600.0, "degrees / second")

a = (n / d)**(1/3)



In [ ]:
a.to("micrometers")